# 01. Calibration NPZ

Detects the ArUco marker in one frame of each camera and saves the camera extrinsics (R, t).
Run this only after moving a camera; otherwise reuse the existing NPZ files.

| | |
|---|---|
| Input | `calibration.frame_index` frame of the top and front videos |
| Output | `inputs.calib_top_npz`, `inputs.calib_front_npz`, check images in the results folder |
| GPU | Not required |

In [ ]:
#@title 0. Setup (clone repository, install packages, mount Drive)
REPO_URL = "https://github.com/dodokim777-boop/underwater-gripper-tracking.git"  #@param {type:"string"}

import os, sys, subprocess, warnings
REPO_DIR = '/content/' + REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=False)
%pip install -q -r {REPO_DIR}/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, REPO_DIR)
import pandas as pd
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
print('Ready:', REPO_DIR)

In [ ]:
#@title 1. Settings file
SETTINGS_PATH = "/content/drive/MyDrive/files/experiment_settings.yaml"  #@param {type:"string"}

import shutil
from main_code import settings

if not os.path.exists(SETTINGS_PATH):
    os.makedirs(os.path.dirname(SETTINGS_PATH), exist_ok=True)
    shutil.copy(os.path.join(REPO_DIR, 'experiment_settings.yaml'), SETTINGS_PATH)
    raise RuntimeError(f'Template copied to {SETTINGS_PATH}. Edit the required values and run this cell again.')

settings.configure(SETTINGS_PATH)
settings.print_summary('calibration')
settings.check('calibration')

## 2. Detect markers and save

Check the images: X (red) to the right, Y (green) into the tank, Z (blue) up.
If a marker is not detected, set `calibration.frame_index` to a frame where the marker is clearly visible.

In [ ]:
#@title 2. Detect markers and save NPZ
OVERWRITE = False  #@param {type:"boolean"}

import cv2, matplotlib.pyplot as plt
from preparation.make_calibration_npz import make_calibration_npz

results = {role: make_calibration_npz(role, overwrite=OVERWRITE) for role in ('top', 'front')}

fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
for ax, (role, r) in zip(axes, results.items()):
    ax.imshow(cv2.cvtColor(cv2.imread(r['check_image']), cv2.COLOR_BGR2RGB))
    ax.set_title(role); ax.axis('off')
plt.tight_layout(); plt.show()